[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/information_theory/03_cross_entropy_and_loss_functions/exercises.ipynb)

# Exercises: Cross-Entropy and Loss Functions

**20 fully solved problems** in 4 levels: Concept Check (4), Foundation (6), Applications in AI/ML (6), Challenge (4).

## Level 0 — Concept Check

### Problem L0.1: Cross-Entropy at a Perfect Match

Let $p = q = (\tfrac{1}{2}, \tfrac{1}{4}, \tfrac{1}{4})$. Compute $H(p, q)$ and explain why it is not zero.

**Solution**

When $q = p$, cross-entropy reduces to entropy:

$$
H(p, p) = -\sum_k p_k \log_2 p_k = \tfrac{1}{2}(1) + \tfrac{1}{4}(2) + \tfrac{1}{4}(2) = 1.5 \text{ bits}
$$

It is not zero because even a perfect model must pay the source's intrinsic entropy; only the *excess* $D_{\mathrm{KL}}(p \parallel q) = H(p, q) - H(p)$ vanishes at a perfect match.

$$
\boxed{H(p, p) = H(p) = 1.5 \text{ bits}}
$$

*Key takeaway*: The reachable minimum of cross-entropy is $H(p)$, not zero — losses should be compared to the entropy floor.

### Problem L0.2: One-Hot Cross-Entropy

For a one-hot target $y = (0, 1, 0)$ and prediction $q = (0.2, 0.7, 0.1)$, compute the cross-entropy loss in nats.

**Solution**

With a one-hot target only the true-class term survives:

$$
H(y, q) = -\sum_k y_k \ln q_k = -\ln q_2 = -\ln 0.7 \approx 0.3567 \text{ nats}
$$

$$
\boxed{\mathcal{L} = -\ln 0.7 \approx 0.357 \text{ nats}}
$$

*Key takeaway*: For hard labels, cross-entropy is simply the negative log-probability assigned to the correct class.

### Problem L0.3: The Infinite Penalty

A model assigns $q(x_0) = 0$ to an outcome with $p(x_0) = 0.01$. What is $H(p, q)$, and what practice prevents this in deployed systems?

**Solution**

The term $-p(x_0)\log q(x_0) = -0.01 \log 0 = +\infty$ dominates:

$$
H(p, q) = +\infty
$$

One observed occurrence of a "declared-impossible" event makes the log loss unbounded.
Practical safeguards: additive (Laplace) smoothing $q_k \gets \frac{n_k + \alpha}{N + \alpha K}$, clipping probabilities away from 0, or interpolating with a uniform floor — all guarantee $q \gt 0$ on the whole alphabet.

$$
\boxed{H(p, q) = +\infty; \text{ smoothing keeps } q \gt 0 \text{ everywhere}}
$$

*Key takeaway*: Never assign probability exactly zero to anything that can happen — the log-loss penalty is infinite.

### Problem L0.4: Asymmetry of Cross-Entropy

Let $p = (0.9, 0.1)$ and $q = (0.5, 0.5)$. Compute $H(p, q)$ and $H(q, p)$ in bits and confirm they differ.

**Solution**

$$
H(p, q) = -0.9\log_2 0.5 - 0.1\log_2 0.5 = 0.9 + 0.1 = 1 \text{ bit}
$$

$$
H(q, p) = -0.5\log_2 0.9 - 0.5\log_2 0.1 = 0.5(0.152) + 0.5(3.3219) \approx 1.737 \text{ bits}
$$

Indeed $1 \neq 1.737$: coding a skewed source with a uniform code costs 1 bit, but coding a uniform source with a skewed code is much worse, because the code reserves a long codeword for an outcome that occurs half the time.

$$
\boxed{H(p, q) = 1 \text{ bit} \neq H(q, p) \approx 1.737 \text{ bits}}
$$

*Key takeaway*: The first argument owns the frequencies, the second owns the code — swapping roles changes the bill.

## Level 1 — Foundation

### Problem L1.1: Verify the Decomposition Numerically

For $p = (0.8, 0.2)$ and $q = (0.6, 0.4)$, compute $H(p)$, $H(p, q)$, and $D_{\mathrm{KL}}(p \parallel q)$ in bits, and verify $H(p, q) = H(p) + D_{\mathrm{KL}}(p \parallel q)$.

**Solution**

Entropy:

$$
H(p) = -0.8\log_2 0.8 - 0.2\log_2 0.2 = 0.8(0.3219) + 0.2(2.3219) \approx 0.7219 \text{ bits}
$$

Cross-entropy:

$$
H(p, q) = -0.8\log_2 0.6 - 0.2\log_2 0.4 = 0.8(0.7370) + 0.2(1.3219) \approx 0.8540 \text{ bits}
$$

KL divergence:

$$
D_{\mathrm{KL}}(p \parallel q) = 0.8\log_2\frac{0.8}{0.6} + 0.2\log_2\frac{0.2}{0.4} = 0.8(0.4150) - 0.2(1) \approx 0.1320 \text{ bits}
$$

Check: $0.7219 + 0.1320 = 0.8539 \approx H(p, q)$ (rounding). Verified.

$$
\boxed{H(p, q) \approx 0.854 = H(p) + D_{\mathrm{KL}}(p \parallel q) \approx 0.722 + 0.132 \text{ bits}}
$$

*Key takeaway*: The identity is exact term-by-term algebra: $-p\log q = -p\log p + p\log(p/q)$.

### Problem L1.2: The Optimal Constant Predictor

A binary dataset has 70% positives. Show that the constant prediction minimizing average BCE is $\hat{p} = 0.7$, and compute the resulting loss.

**Solution**

The average loss of a constant $\hat{p}$ is

$$
\mathcal{L}(\hat{p}) = -0.7\ln\hat{p} - 0.3\ln(1 - \hat{p})
$$

Differentiate and set to zero:

$$
\mathcal{L}'(\hat{p}) = -\frac{0.7}{\hat{p}} + \frac{0.3}{1 - \hat{p}} = 0 \implies 0.7(1 - \hat{p}) = 0.3\hat{p} \implies \hat{p} = 0.7
$$

Convexity ($\mathcal{L}'' \gt 0$) confirms a global minimum. The minimal loss is the base-rate entropy:

$$
\mathcal{L}(0.7) = -0.7\ln 0.7 - 0.3\ln 0.3 = H_b(0.3) \approx 0.6109 \text{ nats}
$$

$$
\boxed{\hat{p}^* = 0.7, \quad \mathcal{L}^* = H_b(0.3) \approx 0.611 \text{ nats}}
$$

*Key takeaway*: Under log loss the best constant is the class frequency — properness in action; a model must beat $H_b(\text{base rate})$ to be using its features at all.

### Problem L1.3: Softmax Gradient by Hand

Logits $z = (2, 0, -1)$ with true class 1 (one-hot $y = (1, 0, 0)$). Compute the softmax $q$, the loss in nats, and the gradient $\partial\mathcal{L}/\partial z$.

**Solution**

Exponentials: $(e^2, e^0, e^{-1}) \approx (7.389, 1, 0.368)$; normalizer $Z \approx 8.757$.

$$
q \approx (0.8438, 0.1142, 0.0420)
$$

Loss: $\mathcal{L} = -\ln q_1 \approx -\ln 0.8438 \approx 0.1699$ nats.

Gradient (Theorem C): $\partial\mathcal{L}/\partial z = q - y$:

$$
\frac{\partial\mathcal{L}}{\partial z} \approx (0.8438 - 1, \; 0.1142, \; 0.0420) = (-0.1562, \; 0.1142, \; 0.0420)
$$

Note the components sum to 0 — softmax gradients always live on the zero-sum hyperplane, since adding a constant to all logits leaves the loss unchanged.

$$
\boxed{\mathcal{L} \approx 0.170 \text{ nats}, \quad \nabla_z\mathcal{L} \approx (-0.156, 0.114, 0.042)}
$$

*Key takeaway*: The gradient is prediction minus target — the true class is pushed up by its probability deficit, all others pushed down by their probabilities.

### Problem L1.4: Cross-Entropy Is Linear in $p$, Convex in $q$

Prove that $q \mapsto H(p, q)$ is convex, and that $p \mapsto H(p, q)$ is linear (affine). Which property makes cross-entropy training well-posed?

**Solution**

**Linearity in $p$.** $H(p, q) = \sum_x p(x)\left(-\log q(x)\right)$ is a weighted sum with weights $p(x)$: for fixed $q$ it is a linear functional of $p$. This is what allowed the label-smoothing split $H((1-\epsilon)y + \epsilon u, q) = (1-\epsilon)H(y, q) + \epsilon H(u, q)$.

**Convexity in $q$.** Each map $q \mapsto -\log q(x)$ is convex (second derivative $1/q(x)^2 \gt 0$), and $H(p, q)$ is a nonnegative combination $\sum_x p(x)(-\log q(x))$ of convex functions, hence convex. Strict convexity holds on the support of $p$.

Convexity in $q$ makes the *distribution-space* problem well-posed: minimizing over the simplex has the unique solution $q = p$ (Gibbs). (Composition with a neural network destroys convexity in the *weights*, but the per-example loss surface over logits stays convex.)

$$
\boxed{H(p, q) \text{ is affine in } p \text{ and convex in } q}
$$

*Key takeaway*: The convex leg of cross-entropy is the model's leg — the geometry rewards moving predicted probabilities toward the truth.

### Problem L1.5: Cross-Entropy of Gaussians = MSE

Show that if the model is $q(y \mid x) = \mathcal{N}(y; \mu_\theta(x), \sigma^2)$ with fixed $\sigma$, then minimizing expected cross-entropy (negative log-likelihood) is equivalent to minimizing mean squared error.

**Solution**

The pointwise loss is

$$
-\ln q(y \mid x) = \frac{1}{2}\ln(2\pi\sigma^2) + \frac{(y - \mu_\theta(x))^2}{2\sigma^2}
$$

Averaging over data, the first term is a $\theta$-free constant, so

$$
\arg\min_\theta \mathbb{E}\left[-\ln q(Y \mid X)\right] = \arg\min_\theta \frac{1}{2\sigma^2}\mathbb{E}\left[(Y - \mu_\theta(X))^2\right] = \arg\min_\theta \mathrm{MSE}(\theta)
$$

$$
\boxed{\text{Gaussian NLL with fixed } \sigma \equiv \text{MSE (up to constants)}}
$$

*Key takeaway*: MSE is not an alternative to cross-entropy — it *is* cross-entropy under a homoscedastic Gaussian observation model; every loss encodes a distributional assumption.

### Problem L1.6: The Stable BCE-With-Logits Formula

Derive the numerically stable form of BCE on a raw logit $z$: $\mathcal{L}(z, y) = \max(z, 0) - zy + \ln(1 + e^{-\vert z \vert})$, and explain why the naive form fails at $z = -800$, $y = 1$.

**Solution**

**Derivation.** With $\hat{p} = \sigma(z) = \frac{1}{1 + e^{-z}}$: $-\ln\hat{p} = \ln(1 + e^{-z})$ and $-\ln(1 - \hat{p}) = \ln(1 + e^{z})$, so

$$
\mathcal{L} = y\ln(1 + e^{-z}) + (1 - y)\ln(1 + e^{z}) = \ln(1 + e^{-z}) + (1 - y)z
$$

using $\ln(1 + e^{z}) = z + \ln(1 + e^{-z})$. To keep the exponent nonpositive for either sign of $z$, apply the same identity conditionally, which merges into the single branch-free formula

$$
\mathcal{L}(z, y) = \max(z, 0) - zy + \ln\left(1 + e^{-\vert z \vert}\right)
$$

(check: for $z \ge 0$ it reads $z - zy + \ln(1+e^{-z})$; for $z \lt 0$ it reads $-zy + \ln(1+e^{z})$ — both match the derivation).

**Failure of the naive form.** At $z = -800$: $\sigma(z)$ underflows to exactly $0.0$ in float64, and $-\ln 0 = \infty$ (or NaN after $0 \cdot \infty$). The stable form gives $0 - (-800)(1) + \ln(1 + e^{-800}) \approx 800$ — the mathematically correct huge-but-finite loss.

$$
\boxed{\mathcal{L}(z, y) = \max(z, 0) - zy + \ln(1 + e^{-\vert z \vert})}
$$

*Key takeaway*: Fusing sigmoid and BCE in log-space avoids both overflow and log-of-zero — the reason `BCEWithLogitsLoss` exists.

## Level 2 — Applications in AI/ML

### Problem L2.1: Perplexity Improvement in Bits

Model A has per-token CE 3.466 nats; model B has 3.120 nats. Compute both perplexities, the bits-per-token saving, and the compression interpretation.

**Solution**

Perplexities: $e^{3.466} \approx 32.0$ and $e^{3.120} \approx 22.6$.

Bits per token: $3.466/\ln 2 \approx 5.0$ and $3.120/\ln 2 \approx 4.5$ — a saving of

$$
\Delta = \frac{3.466 - 3.120}{\ln 2} \approx 0.499 \approx 0.5 \text{ bits/token}
$$

Compression view: an arithmetic coder driven by model B would compress the same text with about $0.5$ fewer bits per token — over a 1B-token corpus, roughly 62 MB smaller.

$$
\boxed{\mathrm{PPL}: 32.0 \to 22.6, \quad \text{saving} \approx 0.5 \text{ bits/token}}
$$

*Key takeaway*: Language-model progress is literally compression progress; every 0.69 nats of CE reduction halves perplexity.

### Problem L2.2: Label Smoothing Targets and Optimal Confidence

With $K = 10$ classes and smoothing $\epsilon = 0.1$, write the smoothed target vector for true class 3, and compute the optimal predicted confidence for the true class and the optimal logit gap.

**Solution**

The smoothed target puts $1 - \epsilon + \epsilon/K$ on the true class and $\epsilon/K$ elsewhere:

$$
\tilde{y}_3 = 0.9 + 0.01 = 0.91, \qquad \tilde{y}_k = 0.01 \text{ for } k \neq 3
$$

By Gibbs, the loss-minimizing prediction is $q^* = \tilde{y}$, so optimal confidence is $0.91$.

Optimal logit gap (softmax inverts to logits up to a constant):

$$
z_{\text{true}} - z_{\text{other}} = \ln\frac{0.91}{0.01} = \ln 91 \approx 4.51
$$

versus $\infty$ for hard targets.

$$
\boxed{q^*_{\text{true}} = 0.91, \quad \text{logit gap} = \ln 91 \approx 4.51}
$$

*Key takeaway*: Smoothing converts "push the logit to infinity" into a finite, calibrated target — better-conditioned gradients and less overconfidence.

### Problem L2.3: Focal Loss vs Cross-Entropy on Easy and Hard Examples

Focal loss is $\mathrm{FL}(q_t) = -(1 - q_t)^{\gamma}\log q_t$ where $q_t$ is the true-class probability. For $\gamma = 2$, compare FL to CE at $q_t = 0.9$ (easy) and $q_t = 0.1$ (hard), and compute the down-weighting factors.

**Solution**

Cross-entropy values: $-\ln 0.9 \approx 0.1054$ and $-\ln 0.1 \approx 2.3026$ nats.

Focal modulating factors $(1 - q_t)^2$: easy example $(0.1)^2 = 0.01$; hard example $(0.9)^2 = 0.81$.

Focal losses: $0.01 \times 0.1054 \approx 0.00105$ and $0.81 \times 2.3026 \approx 1.865$.

Relative emphasis: the hard/easy loss ratio grows from $2.3026/0.1054 \approx 22$ (CE) to $1.865/0.00105 \approx 1770$ (FL) — an 81-fold reweighting toward hard examples.

$$
\boxed{\text{easy: } 0.105 \to 0.001, \quad \text{hard: } 2.303 \to 1.865; \quad \text{hard/easy ratio } 22 \to 1770}
$$

*Key takeaway*: Focal loss multiplies CE by $(1-q_t)^{\gamma}$, silencing the flood of well-classified background examples so rare hard positives dominate the gradient — the key to one-stage object detection.

### Problem L2.4: Distillation Gradients and Temperature

In knowledge distillation, the student minimizes CE against the teacher's tempered softmax $p^{(T)}_k \propto e^{u_k/T}$ using its own tempered softmax $q^{(T)}$. Show the student's logit gradient is $\frac{1}{T}\left(q^{(T)} - p^{(T)}\right)$ and explain the standard $T^2$ loss rescaling.

**Solution**

The loss is $\mathcal{L} = -\sum_k p^{(T)}_k \log q^{(T)}_k$ with $q^{(T)} = \mathrm{softmax}(z/T)$.
Apply the softmax-CE gradient theorem to the scaled logits $\tilde{z} = z/T$ (targets $p^{(T)}$ sum to 1):

$$
\frac{\partial\mathcal{L}}{\partial \tilde{z}_k} = q^{(T)}_k - p^{(T)}_k \quad \Longrightarrow \quad \frac{\partial\mathcal{L}}{\partial z_k} = \frac{1}{T}\left(q^{(T)}_k - p^{(T)}_k\right)
$$

Moreover, for large $T$ the tempered softmaxes flatten toward uniform and a first-order expansion gives $q^{(T)}_k - p^{(T)}_k \approx \frac{z_k - u_k}{TK}$ (for zero-mean logits), so the full gradient magnitude scales like $1/T^2$.
Multiplying the distillation term by $T^2$ keeps its gradient magnitude comparable to the hard-label CE term as $T$ is tuned.

$$
\boxed{\nabla_z\mathcal{L} = \tfrac{1}{T}(q^{(T)} - p^{(T)}); \text{ rescale by } T^2 \text{ to balance objectives}}
$$

*Key takeaway*: Higher temperature softens both distributions, exposing the teacher's inter-class similarity structure ("dark knowledge") while shrinking gradients — the $T^2$ factor compensates exactly to first order.

### Problem L2.5: Loss Floor Auditing

A 100-class dataset is known to have 20% uniformly mislabeled examples (mislabels drawn uniformly from the 99 wrong classes). A model reports test CE of 0.30 nats. Estimate the noise-induced loss floor and diagnose the report.

**Solution**

For a clean-deterministic task corrupted this way, the best possible predictor knows the true class and predicts the *noisy-label* distribution: probability $0.8$ on the true class and $0.2/99$ on each other class. Its expected CE per example is the entropy of that distribution:

$$
H = -0.8\ln 0.8 - 0.2\ln\frac{0.2}{99} = 0.8(0.2231) + 0.2\left(\ln 495\right) \approx 0.1785 + 0.2(6.2046) \approx 1.419 \text{ nats}
$$

The reported 0.30 nats is *far below* the 1.419-nat floor — impossible for honest evaluation against noisy labels. Either the test labels are cleaner than claimed, the noise is not uniform/independent, or (most commonly) the model was evaluated on data it memorized, i.e., test-set leakage.

$$
\boxed{\text{floor} \approx 1.42 \text{ nats} \gt 0.30 \text{ reported} \implies \text{leakage or wrong noise model}}
$$

*Key takeaway*: Conditional-entropy floors turn loss numbers into audits — a loss below the information-theoretic floor is a red flag, not an achievement.

### Problem L2.6: Class-Weighted Cross-Entropy

A binary task has 1 positive per 100 examples. With weighted BCE $\mathcal{L} = -w_+ y\ln\hat{p} - w_-(1-y)\ln(1-\hat{p})$ and weights $w_+ = 100, w_- = 1$, find the constant prediction that minimizes expected weighted loss, and compare with the unweighted optimum.

**Solution**

Expected weighted loss for constant $\hat{p}$ with $P(y{=}1) = 0.01$:

$$
\mathcal{L}(\hat{p}) = -0.01 \times 100 \ln\hat{p} - 0.99 \times 1 \ln(1 - \hat{p}) = -\ln\hat{p} - 0.99\ln(1-\hat{p})
$$

Stationarity:

$$
\frac{1}{\hat{p}} = \frac{0.99}{1 - \hat{p}} \implies 1 - \hat{p} = 0.99\hat{p} \implies \hat{p} = \frac{1}{1.99} \approx 0.5025
$$

The unweighted optimum is the base rate $0.01$. Weighting by inverse frequency moves the optimal constant to $\approx 0.5$ — the model behaves as if classes were balanced.

$$
\boxed{\hat{p}^*_{\text{weighted}} = \tfrac{1}{1.99} \approx 0.503 \text{ vs } \hat{p}^*_{\text{unweighted}} = 0.01}
$$

*Key takeaway*: Class weights implicitly re-tilt the prior; predictions become calibrated for the *reweighted* distribution and must be corrected (or thresholds moved) if true-prior probabilities are needed.

## Level 3 — Challenge

### Problem L3.1: Brier Score Is Proper, Squared Error on the Report Is Not Local

The Brier score for report $q$ on a $K$-outcome event is $S(q, x) = \sum_k (q_k - \mathbf{1}\{x = k\})^2$. Prove it is strictly proper, and show it is not local.

**Solution**

**Propriety.** Expected score under truth $p$:

$$
\mathbb{E}_p\left[S(q, X)\right] = \sum_k \mathbb{E}\left[(q_k - \mathbf{1}\{X = k\})^2\right] = \sum_k \left(q_k^2 - 2q_k p_k + p_k\right)
$$

using $\mathbb{E}[\mathbf{1}\{X=k\}] = p_k$ and $\mathbf{1}^2 = \mathbf{1}$. Subtract the score of the honest report:

$$
\mathbb{E}_p\left[S(q, X)\right] - \mathbb{E}_p\left[S(p, X)\right] = \sum_k \left(q_k^2 - 2q_k p_k + p_k^2\right) = \sum_k (q_k - p_k)^2 = \Vert q - p \Vert_2^2 \ge 0
$$

with equality iff $q = p$: strictly proper.

**Non-locality.** $S(q, x)$ depends on the entire vector $q$ (all $q_k^2$ terms), not only on $q_x$. Two reports agreeing on $q_x$ but differing elsewhere receive different scores when $x$ occurs — violating locality. By Bernardo's theorem, no smooth strictly proper local rule other than affine transforms of $-\log q_x$ exists for $K \ge 3$.

$$
\boxed{\mathbb{E}\left[\text{Brier}(q)\right] - \mathbb{E}\left[\text{Brier}(p)\right] = \Vert q - p \Vert_2^2; \text{ proper but not local}}
$$

*Key takeaway*: Properness admits many rules (log, Brier, spherical); locality singles out the logarithm — and with it, information theory.

### Problem L3.2: The Optimal Code-Length Assignment (Kraft-Constrained Optimization)

Codeword lengths $\ell_x$ of a prefix code satisfy Kraft's inequality $\sum_x 2^{-\ell_x} \le 1$. Treating $\ell_x$ as real numbers, prove that expected length $\sum_x p(x)\ell_x$ is minimized at $\ell_x^* = -\log_2 p(x)$, with minimum $H(p)$, and interpret the excess for any other choice.

**Solution**

**Step 1.** At the optimum, Kraft holds with equality (otherwise all lengths could be shrunk). Define implicit probabilities $q(x) = 2^{-\ell_x}$, so $\sum_x q(x) = 1$ and $\ell_x = -\log_2 q(x)$.

**Step 2.** The objective becomes the cross-entropy

$$
\sum_x p(x)\ell_x = -\sum_x p(x)\log_2 q(x) = H(p, q) = H(p) + D_{\mathrm{KL}}(p \parallel q)
$$

**Step 3.** By Gibbs' inequality the minimum over $q$ (equivalently over Kraft-tight length assignments) is at $q = p$:

$$
\ell_x^* = -\log_2 p(x), \qquad \min \mathbb{E}[\ell] = H(p)
$$

**Step 4 (interpretation).** Any other length assignment corresponds to some $q \neq p$ and pays exactly $D_{\mathrm{KL}}(p \parallel q)$ extra bits per symbol. Integer-length constraints (Huffman) cost at most 1 extra bit: $H(p) \le \mathbb{E}[\ell_{\text{Huffman}}] \lt H(p) + 1$.

$$
\boxed{\ell^*(x) = -\log_2 p(x), \quad \text{excess bits} = D_{\mathrm{KL}}(p \parallel q)}
$$

*Key takeaway*: "Cross-entropy = expected code length under the wrong code" is not an analogy — it is a change of variables between codes and distributions via Kraft equality.

### Problem L3.3: Kelly Betting and the Cross-Entropy of Wealth

A gambler repeatedly bets fractions $b_k$ of wealth on a $K$-outcome race with true probabilities $p$ and fair odds $K$-for-1. Prove the asymptotic growth rate of wealth is $W(b) = \log K - H(p, b)$, maximized by $b^* = p$ ("bet your beliefs"), and that using beliefs $q \neq p$ loses exactly $D_{\mathrm{KL}}(p \parallel q)$ per race.

**Solution**

**Step 1.** After each race, wealth multiplies by $K b_x$ when outcome $x$ occurs. After $n$ i.i.d. races,

$$
\frac{1}{n}\log \frac{V_n}{V_0} = \frac{1}{n}\sum_{i=1}^{n} \log\left(K b_{X_i}\right) \xrightarrow{\text{LLN}} \mathbb{E}_p\left[\log K b_X\right] = \log K + \sum_x p(x)\log b_x
$$

**Step 2.** The growth rate is therefore

$$
W(b) = \log K - H(p, b)
$$

Maximizing over the simplex is minimizing cross-entropy: by Gibbs, $b^* = p$, giving $W^* = \log K - H(p)$.

**Step 3.** Betting according to a model $q$ instead loses

$$
W(p) - W(q) = H(p, q) - H(p) = D_{\mathrm{KL}}(p \parallel q)
$$

per race — the gambler pays the KL gap in exponential growth, doubling-rate currency.

$$
\boxed{W(b) = \log K - H(p, b), \quad b^* = p, \quad \text{loss for using } q = D_{\mathrm{KL}}(p \parallel q)}
$$

*Key takeaway*: Kelly's theorem makes the CE decomposition financially literal: entropy is the house's take, KL is the price of wrong beliefs — and log-loss evaluation is a betting market.

### Problem L3.4: Calibration–Sharpness Decomposition of Log Loss

Let a forecaster emit predictions $Q \in [0,1]$ for a binary event $Y$, and let $c(q) = P(Y{=}1 \mid Q{=}q)$ be the calibration function. Prove the expected log loss decomposes as

$$
\mathbb{E}\left[\mathrm{BCE}\right] = \underbrace{\mathbb{E}_Q\left[D_{\mathrm{KL}}\big(c(Q) \parallel Q\big)\right]}_{\text{calibration}} + \underbrace{\mathbb{E}_Q\left[H_b\big(c(Q)\big)\right]}_{\text{refinement}}
$$

where the KL is between Bernoulli distributions, and interpret both terms.

**Solution**

**Step 1.** Condition on the emitted value $Q = q$. Given $Q = q$, $Y$ is Bernoulli with parameter $c(q)$, and the loss scores the constant report $q$:

$$
\mathbb{E}\left[\mathrm{BCE} \mid Q = q\right] = -c(q)\ln q - (1 - c(q))\ln(1 - q) = H_b\big(c(q), q\big)
$$

— a Bernoulli *cross*-entropy between truth $c(q)$ and report $q$.

**Step 2.** Apply the CE decomposition to the Bernoulli pair:

$$
H_b\big(c(q), q\big) = H_b\big(c(q)\big) + D_{\mathrm{KL}}\big(\mathrm{Ber}(c(q)) \parallel \mathrm{Ber}(q)\big)
$$

**Step 3.** Take the expectation over $Q$ (tower property):

$$
\mathbb{E}\left[\mathrm{BCE}\right] = \mathbb{E}_Q\left[D_{\mathrm{KL}}\big(c(Q) \parallel Q\big)\right] + \mathbb{E}_Q\left[H_b\big(c(Q)\big)\right]
$$

**Interpretation.**

- The **calibration** term vanishes iff $c(q) = q$ almost surely — predictions mean what they say; temperature scaling attacks this term only.
- The **refinement** term is the residual Bernoulli entropy after conditioning on the forecast; it is small when forecasts separate the classes sharply, and is lower-bounded by $H(Y \mid X)$ for any forecaster using features $X$.

$$
\boxed{\log\text{ loss} = \text{calibration KL} + \text{refinement entropy}}
$$

*Key takeaway*: Log loss audits two virtues at once — honesty (calibration) and knowledge (sharpness); a model can trade one against the other, and this decomposition exposes the ledger.